In [1]:
!pip install tensorflow matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt
import numpy as np

2026-05-06 20:30:07.351805: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 20:30:07.382031: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
import zipfile

zip_path = "archive (4).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("cats_dogs_dataset")

print("Dataset Extracted Successfully")

Dataset Extracted Successfully


In [ ]:
# =========================
# IMPORTS
# =========================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt
import numpy as np

# =========================
# DATASET PATH
# =========================
dataset_path = "cats_dogs_dataset"

# =========================
# IMAGE SETTINGS
# =========================
img_height = 128
img_width = 128
batch_size = 32

# =========================
# LOAD DATASET
# =========================
dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="both",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

train_ds, val_ds = dataset

# =========================
# CLASS NAMES
# =========================
class_names = train_ds.class_names

print("Classes:", class_names)

# =========================
# PERFORMANCE OPTIMIZATION
# =========================
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# =========================
# BUILD CNN MODEL
# =========================
model = keras.Sequential([

    keras.Input(shape=(128, 128, 3)),

    layers.Rescaling(1./255),

    # Convolution Layer 1
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Convolution Layer 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Convolution Layer 3
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Flatten
    layers.Flatten(),

    # Dense Layer
    layers.Dense(128, activation='relu'),

    # Dropout
    layers.Dropout(0.5),

    # Output Layer
    layers.Dense(1, activation='sigmoid')
])

# =========================
# COMPILE MODEL
# =========================
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# MODEL SUMMARY
# =========================
model.summary()

# =========================
# CHECKPOINT
# =========================
checkpoint = keras.callbacks.ModelCheckpoint(
    "best_cnn_model.keras",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# =========================
# TRAIN MODEL
# =========================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint]
)

# =========================
# EVALUATE MODEL
# =========================
loss, accuracy = model.evaluate(val_ds)

print("\nValidation Accuracy:", accuracy)

# =========================
# LOAD BEST MODEL
# =========================
best_model = keras.models.load_model("best_cnn_model.keras")

# =========================
# FINAL EVALUATION
# =========================
best_loss, best_acc = best_model.evaluate(val_ds)

print("\nBest Saved Model Accuracy:", best_acc)

Found 1000 files belonging to 2 classes.
Using 800 files for training.
Using 200 files for validation.
Classes: ['cats_set', 'dogs_set']


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_3 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20

Epoch 1: val_accuracy improved from None to 0.50500, saving model to best_cnn_model.keras
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 116ms/step - accuracy: 0.4837 - loss: 0.7034 - val_accuracy: 0.5050 - val_loss: 0.6954
Epoch 2/20

Epoch 2: val_accuracy improved from 0.50500 to 0.56500, saving model to best_cnn_model.keras
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 118ms/step - accuracy: 0.5888 - loss: 0.6798 - val_accuracy: 0.5650 - val_loss: 0.6960
Epoch 3/20
20/25 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.6182 - loss: 0.6502